# DistilBERT fine-tuning on SBERT-labeled captions

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 3.3.3, 4.3 (Table 8).

Fine-tunes distilbert-base-uncased on the CoNeTTE captions labeled by the SBERT danger scorer (low threshold), for six configurations (random / alignment-stratified split × standard CE / weighted CE / focal loss). The selected configuration is alignment-stratified split with weighted cross-entropy.

**Environment.** Written for Google Colab (NVIDIA T4 GPU, Hugging Face transformers); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** Caption dataset with SBERT danger labels (data/captions/).

**Outputs.** Fine-tuned models and caption test-set metrics.


# DistilBERT Low-Threshold Experiments

This notebook fine-tunes DistilBERT on the low-threshold SBERT-scored dataset only.

Training label source: `sbert_danger_label`

Experiments:

- random split + standard cross entropy
- random split + weighted cross entropy
- random split + focal loss
- alignment-stratified split + standard cross entropy
- alignment-stratified split + weighted cross entropy
- alignment-stratified split + focal loss


## 1. Install Dependencies

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

## 2. Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 3. Mount Drive and Configure Paths

Edit these paths if your Drive folder is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
INPUT_CSV_PATH = Path(
    '/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/SBERT/Danger Dataset/outputs/danger_non_danger_sentencebert_semantic_scored_low_threshold.csv'
)

OUTPUTS_DIR = Path(
    '/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/DIstilBERT/training/low_threshold_experiments'
)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('INPUT_CSV_PATH =', INPUT_CSV_PATH)
print('OUTPUTS_DIR =', OUTPUTS_DIR)

## 4. Training Config

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
TRAIN_SIZE = 0.60
VAL_SIZE = 0.20
TEST_SIZE = 0.20
RANDOM_STATE = 42
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MIN_DANGER_RECALL = 0.80

LABEL_SOURCE_COL = 'sbert_danger_label'
ALIGNMENT_COL = 'sbert_class_alignment_score'
LABEL_MAP = {'no_danger_risk': 0, 'danger_risk': 1}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

## 5. Load Dataset

The low-threshold semantic-scored CSV has complete labels in `sbert_danger_label`, so this notebook copies that column into `semantic_label_final` for DistilBERT training.

In [ ]:
def load_dataset() -> pd.DataFrame:
    df = pd.read_csv(INPUT_CSV_PATH)

    required = ['filename', 'caption', LABEL_SOURCE_COL]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Input CSV is missing columns: {missing}')

    df = df.dropna(subset=['filename', 'caption', LABEL_SOURCE_COL]).copy()
    df['filename'] = df['filename'].astype(str).str.strip()
    df['caption'] = df['caption'].astype(str).str.strip()
    df[LABEL_SOURCE_COL] = df[LABEL_SOURCE_COL].astype(str).str.strip()
    df = df[(df['filename'] != '') & (df['caption'] != '')]
    df = df[df[LABEL_SOURCE_COL].isin(LABEL_MAP)].copy()

    df['semantic_label_final'] = df[LABEL_SOURCE_COL]
    df['label'] = df['semantic_label_final'].map(LABEL_MAP).astype(int)

    print('rows:', len(df))
    print('\nlabels:')
    print(df['semantic_label_final'].value_counts().to_string())

    if ALIGNMENT_COL in df.columns:
        print('\nclass alignment:')
        print(df[ALIGNMENT_COL].value_counts(dropna=False).to_string())

    return df.reset_index(drop=True)

df = load_dataset()
df.head()

## 6. Split Helper

`random` stratifies by danger label.

`alignment_stratified` stratifies by danger label plus SBERT class-alignment score.

In [ ]:
def split_dataset(
    df: pd.DataFrame,
    split_mode: str,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if split_mode not in {'random', 'alignment_stratified'}:
        raise ValueError('split_mode must be random or alignment_stratified')

    working = df.copy()
    if split_mode == 'alignment_stratified' and ALIGNMENT_COL in working.columns:
        alignment = (
            pd.to_numeric(working[ALIGNMENT_COL], errors='coerce')
            .fillna(-1)
            .astype(int)
            .astype(str)
        )
        working['_stratify_key'] = working['label'].astype(str) + '__' + alignment
    else:
        working['_stratify_key'] = working['label'].astype(str)

    key_counts = working['_stratify_key'].value_counts()
    valid_keys = key_counts[key_counts >= 2].index
    valid_df = working[working['_stratify_key'].isin(valid_keys)].copy()
    leftover = working[~working['_stratify_key'].isin(valid_keys)].copy()

    train_df, temp_df = train_test_split(
        valid_df,
        test_size=(VAL_SIZE + TEST_SIZE),
        random_state=random_state,
        stratify=valid_df['_stratify_key'],
    )

    temp_key_counts = temp_df['_stratify_key'].value_counts()
    temp_valid_keys = temp_key_counts[temp_key_counts >= 2].index
    temp_valid = temp_df[temp_df['_stratify_key'].isin(temp_valid_keys)].copy()
    temp_leftover = temp_df[~temp_df['_stratify_key'].isin(temp_valid_keys)].copy()

    val_df, test_df = train_test_split(
        temp_valid,
        test_size=0.5,
        random_state=random_state,
        stratify=temp_valid['_stratify_key'],
    )

    if not leftover.empty:
        train_df = pd.concat([train_df, leftover], ignore_index=True)
    if not temp_leftover.empty:
        half = len(temp_leftover) // 2
        val_df = pd.concat([val_df, temp_leftover.iloc[:half]], ignore_index=True)
        test_df = pd.concat([test_df, temp_leftover.iloc[half:]], ignore_index=True)

    for split_df in (train_df, val_df, test_df):
        split_df.drop(columns=['_stratify_key'], errors='ignore', inplace=True)

    train_df = train_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f'\nSplit: {split_mode}')
    for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        n_danger = int(split_df['label'].sum())
        n_no_danger = int((split_df['label'] == 0).sum())
        print(f'  {name}: {len(split_df)} | danger={n_danger} | no_danger={n_no_danger}')

    return train_df, val_df, test_df

## 7. Tokenization and Metrics

In [ ]:
def tokenize_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    keep_cols = ['filename', 'caption', 'semantic_label_final', 'label']
    extra_cols = [col for col in ['category', 'source_dataset', ALIGNMENT_COL] if col in df.columns]
    ds = Dataset.from_pandas(df[keep_cols + extra_cols].reset_index(drop=True))

    def tokenize_batch(batch):
        return tokenizer(batch['caption'], truncation=True, max_length=MAX_LENGTH)

    ds = ds.map(tokenize_batch, batched=True)
    model_cols = keep_cols + extra_cols + ['input_ids', 'attention_mask']
    return ds.remove_columns([col for col in ds.column_names if col not in model_cols])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1],
        zero_division=0,
    )
    return {
        'accuracy': accuracy,
        'precision_not_danger_0': precision[0],
        'recall_not_danger_0': recall[0],
        'f1_not_danger_0': f1[0],
        'precision_danger_1': precision[1],
        'recall_danger_1': recall[1],
        'f1_danger_1': f1[1],
        'macro_f1': float((f1[0] + f1[1]) / 2),
    }

## 8. Weighted Cross Entropy and Focal Loss

In [ ]:
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights: list[float], *args, **kwargs):
        super().__init__(*args, **kwargs)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.loss_fn = nn.CrossEntropyLoss(
            weight=torch.tensor(class_weights, dtype=torch.float).to(device)
        )
        print(
            'WeightedLossTrainer: '
            f'no_danger={class_weights[0]:.4f} | danger={class_weights[1]:.4f}'
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


class FocalLossTrainer(Trainer):
    def __init__(
        self,
        class_weights: list[float] | None = None,
        gamma: float = 2.0,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.gamma = gamma
        self.class_weights = None
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
        print('FocalLossTrainer:', f'gamma={self.gamma}', f'class_weights={class_weights}')

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        ce_loss = F.cross_entropy(
            outputs.logits,
            labels,
            weight=self.class_weights,
            reduction='none',
        )
        pt = torch.exp(-ce_loss)
        loss = ((1.0 - pt) ** self.gamma * ce_loss).mean()
        return (loss, outputs) if return_outputs else loss


def compute_class_weights(train_df: pd.DataFrame) -> list[float]:
    total = len(train_df)
    n_no_danger = int((train_df['label'] == 0).sum())
    n_danger = int((train_df['label'] == 1).sum())
    if n_no_danger == 0 or n_danger == 0:
        raise ValueError('Both classes must be present to compute class weights.')
    return [total / (2 * n_no_danger), total / (2 * n_danger)]

## 9. Prediction Output Helper

In [ ]:
def save_prediction_outputs(
    trainer: Trainer,
    dataset,
    source_df: pd.DataFrame,
    split_name: str,
    exp_dir: Path,
) -> tuple[dict, np.ndarray]:
    predictions = trainer.predict(dataset)
    pred_labels = np.argmax(predictions.predictions, axis=-1)

    pred_df = source_df[
        ['filename', 'caption', 'semantic_label_final', 'label']
    ].reset_index(drop=True).copy()
    pred_df['prediction'] = pred_labels
    pred_df['prediction_name'] = [INV_LABEL_MAP[int(idx)] for idx in pred_labels]
    pred_df.to_csv(exp_dir / f'{split_name}_predictions.csv', index=False)

    cm = confusion_matrix(pred_df['label'], pred_df['prediction'], labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=['true_0_not_danger', 'true_1_danger'],
        columns=['pred_0_not_danger', 'pred_1_danger'],
    )
    cm_df.to_csv(exp_dir / f'confusion_matrix_{split_name}.csv')

    precision, recall, f1, support = precision_recall_fscore_support(
        pred_df['label'],
        pred_df['prediction'],
        labels=[0, 1],
        zero_division=0,
    )
    per_class_df = pd.DataFrame(
        {
            'class_label': ['no_danger_risk (0)', 'danger_risk (1)'],
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'support': support,
        }
    )
    per_class_df.to_csv(exp_dir / f'per_class_{split_name}_metrics.csv', index=False)

    with (exp_dir / f'{split_name}_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(predictions.metrics, handle, indent=2)

    return predictions.metrics, cm

## 10. Run One Experiment

In [ ]:
def run_experiment(
    experiment_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    loss_name: str,
) -> dict:
    exp_dir = OUTPUTS_DIR / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"=" * 72}')
    print('EXPERIMENT:', experiment_name)
    print('loss:', loss_name)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds = tokenize_dataset(train_df, tokenizer)
    val_ds = tokenize_dataset(val_df, tokenizer)
    test_ds = tokenize_dataset(test_df, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    training_args = TrainingArguments(
        output_dir=str(exp_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='recall_danger_1',
        greater_is_better=True,
        save_total_limit=2,
        report_to='none',
        logging_steps=50,
    )

    common_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': train_ds,
        'eval_dataset': val_ds,
        'data_collator': DataCollatorWithPadding(tokenizer=tokenizer),
        'compute_metrics': compute_metrics,
    }

    if loss_name == 'weighted_ce':
        trainer = WeightedLossTrainer(
            class_weights=compute_class_weights(train_df),
            **common_kwargs,
        )
    elif loss_name == 'focal_loss':
        trainer = FocalLossTrainer(
            class_weights=compute_class_weights(train_df),
            gamma=2.0,
            **common_kwargs,
        )
    elif loss_name == 'standard_ce':
        trainer = Trainer(**common_kwargs)
    else:
        raise ValueError(f'Unknown loss_name: {loss_name}')

    trainer.train()
    trainer.save_model(str(exp_dir / 'best_model'))
    tokenizer.save_pretrained(str(exp_dir / 'tokenizer'))

    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(exp_dir / 'training_history.csv', index=False)

    val_metrics = trainer.evaluate(eval_dataset=val_ds)
    with (exp_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(val_metrics, handle, indent=2)

    _, val_cm = save_prediction_outputs(trainer, val_ds, val_df, 'val', exp_dir)
    test_metrics, test_cm = save_prediction_outputs(trainer, test_ds, test_df, 'test', exp_dir)

    print(
        'VAL:',
        f'danger_recall={val_metrics.get("eval_recall_danger_1", 0):.4f}',
        f'danger_f1={val_metrics.get("eval_f1_danger_1", 0):.4f}',
        f'macro_f1={val_metrics.get("eval_macro_f1", 0):.4f}',
        f'accuracy={val_metrics.get("eval_accuracy", 0):.4f}',
    )
    print(f'VAL CM: TN={val_cm[0,0]} FP={val_cm[0,1]} FN={val_cm[1,0]} TP={val_cm[1,1]}')
    print(f'TEST CM: TN={test_cm[0,0]} FP={test_cm[0,1]} FN={test_cm[1,0]} TP={test_cm[1,1]}')
    print('saved:', exp_dir)

    return {
        'experiment_name': experiment_name,
        'loss': loss_name,
        'accuracy': val_metrics.get('eval_accuracy'),
        'precision_not_danger_0': val_metrics.get('eval_precision_not_danger_0'),
        'recall_not_danger_0': val_metrics.get('eval_recall_not_danger_0'),
        'f1_not_danger_0': val_metrics.get('eval_f1_not_danger_0'),
        'precision_danger_1': val_metrics.get('eval_precision_danger_1'),
        'recall_danger_1': val_metrics.get('eval_recall_danger_1'),
        'f1_danger_1': val_metrics.get('eval_f1_danger_1'),
        'macro_f1': val_metrics.get('eval_macro_f1'),
        'eval_loss': val_metrics.get('eval_loss'),
        'tn': int(val_cm[0, 0]),
        'fp': int(val_cm[0, 1]),
        'fn': int(val_cm[1, 0]),
        'tp': int(val_cm[1, 1]),
        'test_tn': int(test_cm[0, 0]),
        'test_fp': int(test_cm[0, 1]),
        'test_fn': int(test_cm[1, 0]),
        'test_tp': int(test_cm[1, 1]),
        'test_accuracy': test_metrics.get('test_accuracy'),
        'test_recall_danger_1': test_metrics.get('test_recall_danger_1'),
        'test_f1_danger_1': test_metrics.get('test_f1_danger_1'),
        'test_macro_f1': test_metrics.get('test_macro_f1'),
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df),
        'train_danger': int(train_df['label'].sum()),
        'train_no_danger': int((train_df['label'] == 0).sum()),
        'experiment_dir': str(exp_dir),
    }

## 11. Run Experiments

This cell runs all six experiments. It can take a while on Colab.

In [ ]:
results = []

for split_mode in ['random', 'alignment_stratified']:
    train_df, val_df, test_df = split_dataset(df, split_mode=split_mode)

    split_dir = OUTPUTS_DIR / f'split_{split_mode}'
    split_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(split_dir / 'train_split.csv', index=False)
    val_df.to_csv(split_dir / 'val_split.csv', index=False)
    test_df.to_csv(split_dir / 'test_split.csv', index=False)

    for loss_name in ['standard_ce', 'weighted_ce', 'focal_loss']:
        experiment_name = f'low_threshold__{split_mode}__{loss_name}'
        result = run_experiment(
            experiment_name=experiment_name,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            loss_name=loss_name,
        )
        results.append(result)

len(results)

## 12. Compare Experiments

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df['meets_recall_threshold'] = comparison_df['recall_danger_1'] >= MIN_DANGER_RECALL
comparison_df = comparison_df.sort_values(
    ['meets_recall_threshold', 'f1_danger_1', 'macro_f1', 'recall_danger_1'],
    ascending=[False, False, False, False],
).reset_index(drop=True)
comparison_df['rank'] = comparison_df.index + 1

comparison_dir = OUTPUTS_DIR / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(
    comparison_dir / 'low_threshold_experiments_comparison.csv',
    index=False,
)

thesis_cols = [
    'rank',
    'experiment_name',
    'loss',
    'meets_recall_threshold',
    'recall_danger_1',
    'f1_danger_1',
    'macro_f1',
    'accuracy',
    'precision_danger_1',
    'recall_not_danger_0',
    'f1_not_danger_0',
    'tn',
    'fp',
    'fn',
    'tp',
    'test_recall_danger_1',
    'test_f1_danger_1',
    'test_macro_f1',
    'test_tn',
    'test_fp',
    'test_fn',
    'test_tp',
    'train_size',
    'train_danger',
    'train_no_danger',
    'experiment_dir',
]
comparison_df[thesis_cols].to_csv(
    comparison_dir / 'low_threshold_experiments_thesis_table.csv',
    index=False,
)

print('comparison saved:', comparison_dir)
comparison_df[thesis_cols]

## 13. Best Model

In [ ]:
eligible = comparison_df[comparison_df['meets_recall_threshold']]
best = eligible.iloc[0] if not eligible.empty else comparison_df.iloc[0]

print('BEST MODEL:', best['experiment_name'])
print(
    f"danger_recall={best['recall_danger_1']:.4f} | "
    f"danger_f1={best['f1_danger_1']:.4f} | "
    f"macro_f1={best['macro_f1']:.4f}"
)
print('model dir:', best['experiment_dir'])